In [ ]:
import os

from BudaOCR.Config import CHARSET
from BudaOCR.Networks import Easter2ViTNetwork
from BudaOCR.Encoder import WylieEncoder
from BudaOCR.Trainer import OCRTrainer

from BudaOCR.Utils import (
    build_data_paths,
    create_dir,
    shuffle_data
    )

In [ ]:
from huggingface_hub import snapshot_download
data_path = snapshot_download(repo_id="BDRC/Karmapa8", repo_type="dataset",  cache_dir="Datasets", force_download=True)

In [ ]:
# local dir
wylie_encoder = WylieEncoder(CHARSET)

dataset_path = "Datasets/Karmapa8"
image_paths, label_paths = build_data_paths(dataset_path, img_file_ext="jpg")
image_paths, label_paths = shuffle_data(image_paths, label_paths)

print(f"Images: {len(image_paths)}, Labels: {len(label_paths)}")

output_dir = os.path.join("Output")
create_dir(output_dir)

In [ ]:
image_width = 3200
image_height = 100
encoder = wylie_encoder
num_classes = encoder.num_classes()

network = Easter2ViTNetwork(image_width, image_height, num_classes=num_classes)
batch_size = 32
workers = 4

In [ ]:
ocr_trainer = OCRTrainer(
    network=network,
    label_encoder=encoder,
    workers=workers, 
    image_width=image_width,
    image_height=image_height,
    batch_size=32, 
    output_dir=output_dir, 
    preload_labels=True
    )

In [ ]:
ocr_trainer.init(image_paths, label_paths)


In [ ]:
num_epochs = 24
scheduler_start = 2
ocr_trainer.train(epochs=num_epochs, check_cer=True, export_onnx=True, silent=False)